# บ้านดีมาร์ท — วิเคราะห์ยอดขายและประสิทธิภาพช่องทางการขาย

**Baan Dee Mart — Thai E-commerce Sales Analytics**

> ข้อมูลนี้เป็น **Synthetic Dataset** เพื่อ Portfolio ไม่ใช่ข้อมูลจากบริษัทจริง

Notebook นี้ตอบคำถามธุรกิจด้วย Python/pandas ตั้งแต่ตรวจข้อมูล ทำความสะอาด คำนวณ Metrics วิเคราะห์ลูกค้า สร้างกราฟ และสรุปข้อเสนอแนะ โดยหนึ่งแถวคือสินค้า 1 รายการในคำสั่งซื้อ 1 รายการ

## Phase 1: Business Understanding

บ้านดีมาร์ทเป็นร้านค้าออนไลน์ SME ไทยที่ขายผ่าน Shopee, Lazada, TikTok Shop, Facebook และ Website เจ้าของร้านต้องการรู้ช่องทาง สินค้า พื้นที่ และลูกค้าที่ควรจัดลำดับความสำคัญ

**ข้อจำกัด:** เป็นข้อมูลจำลองปีเดียว ไม่มีต้นทุน กำไร ค่าโฆษณา Traffic และประวัติลูกค้านอกช่วงข้อมูล จึงไม่สรุป Profit หรือ ROI

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / 'data' / 'raw').is_dir()),
    Path.cwd(),
)
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'orders_raw.csv'
CLEAN_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'orders_cleaned.csv'
FIGURES_DIR = PROJECT_ROOT / 'outputs'
raw_df = pd.read_csv(RAW_PATH)
clean_df = pd.read_csv(CLEAN_PATH, parse_dates=['order_date'])
print('Project root:', PROJECT_ROOT)
print('Raw shape:', raw_df.shape)
print('Cleaned shape:', clean_df.shape)

## Phase 2: Data Understanding

ตรวจ shape, ตัวอย่างข้อมูล, data types, missing values, duplicate, unique values, ค่าสถิติ และ Grain ก่อนแก้ไขข้อมูล

In [ ]:
display(raw_df.head())
display(raw_df.dtypes.to_frame('data_type'))
display(raw_df.isna().sum().to_frame('missing_count'))
print('Duplicate rows:', raw_df.duplicated().sum())
display(raw_df[['quantity', 'unit_price', 'discount', 'sales_amount']].describe().T)

## Phase 3: Data Cleaning

ใช้เฉพาะ `Completed` เป็นยอดขาย ตัดวันที่อ่านไม่ได้ duplicate quantity ที่ไม่ถูกต้อง และราคา outlier ตรวจ `sales_amount` กับสูตร `quantity × unit_price × (1 - discount)` โดยไม่เติม missing discount เป็นศูนย์โดยอัตโนมัติ

In [ ]:
print('Missing after cleaning:', clean_df.isna().sum().sum())
print('Duplicates after cleaning:', clean_df.duplicated().sum())
print(clean_df.order_status.value_counts(dropna=False))

## Phase 4-5: Feature Engineering และ Sales Analysis

คำนวณ `gross_sales`, `net_sales`, เดือน, customer type และรวม Metrics ที่ระดับ Order เพื่อไม่ให้นับ Order Line เป็น Order

In [ ]:
orders = clean_df.groupby('order_id', as_index=False).agg(order_date=('order_date','first'), customer_id=('customer_id','first'), channel=('channel','first'), order_value=('net_sales','sum'))
metrics = {
    'ยอดขายสุทธิ': clean_df.net_sales.sum(),
    'คำสั่งซื้อ': orders.order_id.nunique(),
    'ลูกค้า': clean_df.customer_id.nunique(),
    'AOV': orders.order_value.mean(),
    'Median Order Value': orders.order_value.median(),
}
pd.Series(metrics)

In [ ]:
sales_by_channel = clean_df.groupby('channel').net_sales.sum().sort_values(ascending=False)
orders_by_channel = orders.groupby('channel').order_id.nunique().sort_values(ascending=False)
aov_by_channel = orders.groupby('channel').order_value.mean().sort_values(ascending=False)
display(sales_by_channel.to_frame('net_sales'))
display(orders_by_channel.to_frame('order_count'))
display(aov_by_channel.to_frame('aov'))

In [ ]:
sales_by_category = clean_df.groupby('category').net_sales.sum().sort_values(ascending=False)
sales_by_product = clean_df.groupby(['product_id', 'product_name']).net_sales.sum().sort_values(ascending=False)
sales_by_province = clean_df.groupby('province').net_sales.sum().sort_values(ascending=False)
sales_by_region = clean_df.groupby('region').net_sales.sum().sort_values(ascending=False)
monthly_sales = clean_df.groupby('order_month').net_sales.sum()
display(sales_by_category.to_frame('net_sales'))
display(sales_by_product.head(10).to_frame('net_sales'))
display(sales_by_province.head(10).to_frame('net_sales'))
display(sales_by_region.to_frame('net_sales'))
display(monthly_sales.to_frame('net_sales'))

## Phase 6: Customer Analysis

ลูกค้าใหม่หมายถึงลูกค้าที่มี 1 Order ในช่วงข้อมูล และลูกค้าที่ซื้อซ้ำมีมากกว่า 1 Order ไม่ใช่สถานะตลอดอายุลูกค้า

In [ ]:
customer_summary = orders.groupby('customer_id').agg(order_count=('order_id','nunique'), monetary=('order_value','sum'))
customer_summary['customer_type'] = np.where(customer_summary.order_count > 1, 'ลูกค้าที่ซื้อซ้ำ', 'ลูกค้าใหม่')
display(customer_summary.groupby('customer_type').size().to_frame('customer_count'))

## Phase 7-8: Statistical Thinking และ Visualization

Mean สูงกว่า Median อาจสะท้อนคำสั่งซื้อขนาดใหญ่บางรายการ และ Correlation ไม่ได้แปลว่า Causation กราฟภาษาไทย 10 รายการถูกบันทึกไว้ใน `outputs/`

In [ ]:
print('Mean:', orders.order_value.mean())
print('Median:', orders.order_value.median())
print('Std:', orders.order_value.std())
print('75th percentile:', orders.order_value.quantile(.75))
from IPython.display import display, Image
figure_names = [
    '01_monthly_sales_trend.png', '02_sales_by_channel.png', '03_orders_by_channel.png',
    '04_aov_by_channel.png', '05_sales_by_category.png', '06_top_products.png',
    '07_sales_by_region.png', '08_customer_purchase_frequency.png',
    '09_new_vs_repeat_customers.png', '10_order_value_distribution.png'
]
for figure_name in figure_names:
    display(Image(filename=str(FIGURES_DIR / figure_name), width=700))

## Phase 9-10: Business Insights และ Executive Summary

ผลสรุปใน Notebook ต้องอ่านคู่กับ `BUSINESS_INSIGHTS.md`, `EXECUTIVE_SUMMARY.md` และ PDF โดยทุกตัวเลขดึงจาก Cleaned Dataset และทุกข้อเสนอแนะระบุข้อจำกัดไว้ชัดเจน